# What Makes a Mortgage Cost More?: Data Preparation

This notebook prepares the raw data behind the Streamlit app (`app.py`). I downloaded 2025 home-purchase mortgage applications for six states from the HMDA website. I picked two states for each home-price tier: expensive (CA, MA), mid (FL, TX) and low (MI, OH) based on their median values.

I wanted a particular set of comparable factors, so this notebook checks that the downloads are complete and keeps only the loans that fit my question well.

1. Setup
2. Convert the CSV files to parquet
3. Check that every download is complete
4. Load only the columns I need
5. Keep comparable applications (scope filters)
6. Flag exempt lenders and save the file for notebook 2
7. Summary

## Step 1 · Setup

This cell is used to create folders where our data will be saved.

The data is saved in three folders, one for each stage:
- data/raw/- holds the 6 downloaded state files. Nothing is removed from them.
- data/interim/- holds one combined file with only the applications I need.
- data/processed/- holds the final clean file that the app uses. Notebook 2 creates it
- mkdir - creates a folder if it doesn't exist yet.

**To run this on your computer, change `PROJECT_DIR` to your own project folder.**

In [1]:
from pathlib import Path
import pandas as pd

year = 2025
states_chosen = ["CA", "MA", "FL", "TX", "MI", "OH"]
state_tiers = {"CA": "Expensive", "MA": "Expensive", "FL": "Mid",
               "TX": "Mid", "MI": "Low", "OH": "Low"}

# Folders: this notebook sits in the project folder
PROJECT_DIR = Path(r"C:\Users\havil\CMSE_midterm_HMDA_project")
RAW_DIR = PROJECT_DIR / "data" / "raw"          # parquet copies of the CSVs
INTERIM_DIR = PROJECT_DIR / "data" / "interim"  # filtered data
RAW_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_DIR)

Project folder: C:\Users\havil\CMSE_midterm_HMDA_project


## Step 2 · Convert the CSV files to parquet

In this step, the for loop converts each state's CSV file into a parquet file. As the data is around 2 GB for this dataset, using parquet would compress it well and helps with faster processing. i can also pick only the columns I need unlike CSV. Hence, I used parquet.

- dtype=str - reads every column as text, so HMDA codes like "Exempt" aren't lost.
- keep_default_na=False - keeps the text "NA" instead of turning it into a blank, so it can be counted later.
- States already converted are skipped.

In [2]:
for state in states_chosen:
    csv_file = PROJECT_DIR / f"loan_purposes_1_state_{state}.csv"
    parquet_file = RAW_DIR / f"hmda_{year}_{state}_purchase.parquet"

    if parquet_file.exists():
        print(state, " is already converted")
        continue

    # Read every column as text so codes like "NA" and "Exempt" stay as they are
    state_df = pd.read_csv(csv_file, dtype=str, keep_default_na=False)
    state_df.to_parquet(parquet_file, index=False)
    print(state, "converted:", len(state_df), "rows")

CA  is already converted
MA  is already converted
FL  is already converted
TX  is already converted
MI  is already converted
OH  is already converted


## Step 3 · Check the data is complete

Here, I am trying to check if the download was successful.
The total must match the HMDA Data Browser: 2,282,462 applications.

Because the first California and Texas downloads were silently cut off (California had 57,100 of 465,779 rows) but opened without errors. This check caught it, and both were re-downloaded.

In [3]:
EXPECTED_TOTAL_COUNT = 2_282_462   

total = 0
for state in states_chosen:
    rows = len(pd.read_parquet(RAW_DIR / f"hmda_{year}_{state}_purchase.parquet", columns=["state_code"]))
    print(f"{state} ({state_tiers[state]}): {rows:,} applications")
    total += rows

print(f"\nTotal downloaded: {total:,}  |  Expected: {EXPECTED_TOTAL_COUNT:,}")
print("Data download is complete" if total == EXPECTED_TOTAL_COUNT else "Mismatch: re-download the small state")

CA (Expensive): 465,779 applications
MA (Expensive): 105,202 applications
FL (Mid): 561,246 applications
TX (Mid): 734,539 applications
MI (Low): 182,986 applications
OH (Low): 232,710 applications

Total downloaded: 2,282,462  |  Expected: 2,282,462
Data download is complete


## Step 4 · Load only the needed columns

From the above lines of code, the parquet files keep all 99 columns, but only 23 useful ones are loaded here: 15 for the analysis and 8 used only for filtering in the next step (dropped once used).

Using the code below, the six states are stacked into one table, and I check for duplicate rows. 

dtype_backend="pyarrow" stores text compactly. Using pyarrow, an entire column of text is stored together in one compact block, cutting memory use heavily from 5.58 GB to 0.74 GB.

Duplicates are counted in this dataset but not dropped: Because this dataset has no unique applicant ID and rounds loan amounts, so identical rows are very likely another different real loans.

In [4]:
filter_columns = ["loan_purpose", "lien_status", "occupancy_type", "total_units",
               "construction_method", "business_or_commercial_purpose",
               "reverse_mortgage", "open-end_line_of_credit"]

my_columns = [
    "state_code", "action_taken", "loan_type",           
    "interest_rate", "rate_spread", "total_loan_costs",  
    "loan_amount", "loan_to_value_ratio", "property_value",  
    "income", "debt_to_income_ratio",                   
    "loan_term", "intro_rate_period",                    
    "interest_only_payment", "balloon_payment",          
]

# Load each state, then stack them into one table
all_states_table = []
for state in states_chosen:
    file = RAW_DIR / f"hmda_{year}_{state}_purchase.parquet"
    all_states_table.append(pd.read_parquet(file, columns=my_columns + filter_columns, dtype_backend="pyarrow"))
df = pd.concat(all_states_table, ignore_index=True)

print(f"Loaded {len(df):,} rows x {df.shape[1]} columns")

Loaded 2,282,462 rows x 23 columns


## Step 5 · Keep comparable applications

For fair comparison, I am keepign everything with similar rules
- Home purchases- already filtered at download and checked again here
- First liens- because second mortgages cost more
- Principal residences- because investor and vacation-home loans cost more
- 1–4 unit homes- because larger buildings are commercial loans
- Site-built homes- because manufactured homes are financed differently
- Standard loans only- so no business, reverse-mortgage or credit-line products

In [5]:
counts = [("Downloaded", len(df))]

df = df[df["loan_purpose"] == "1"];                          counts.append(("Home purchase", len(df)))
df = df[df["lien_status"] == "1"];                           counts.append(("First lien", len(df)))
df = df[df["occupancy_type"] == "1"];                        counts.append(("Principal residence", len(df)))
df = df[df["total_units"].isin(["1", "2", "3", "4"])];       counts.append(("1-4 units", len(df)))
df = df[df["construction_method"] == "1"];                   counts.append(("Site-built", len(df)))


# 2 = No. Exempt (1111) rows are removed because they can't be confirmed as standard loans
df = df[df["business_or_commercial_purpose"] == "2"]; counts.append(("Not business purpose", len(df)))
df = df[df["reverse_mortgage"] == "2"];               counts.append(("Not reverse mortgage", len(df)))
df = df[df["open-end_line_of_credit"] == "2"];        counts.append(("Not open-end credit line", len(df)))

funnel = pd.DataFrame(counts, columns=["step", "rows_remaining"])
funnel["removed"] = funnel["rows_remaining"].diff().fillna(0).abs().astype(int)
display(funnel)

# Save the funnel for the app's Data & Preparation page
funnel.to_csv(PROJECT_DIR / "data" / "funnel_scope.csv", index=False)

,step,rows_remaining,removed
0,Downloaded,2282462,0
1,Home purchase,2282462,0
2,First lien,2200715,81747
3,Principal residence,1940913,259802
4,1-4 units,1940898,15
5,Site-built,1760505,180393
6,Not business purpose,1737186,23319
7,Not reverse mortgage,1736352,834
8,Not open-end credit line,1727644,8708


## Step 6 · Flag Exempt rows and save

Here, I am copying the data into another dataframe. The 8 columns that were used to filter that data for our needs are noe dropped. 

In [6]:
final_df = df.copy()

final_df = final_df.drop(columns=filter_columns).reset_index(drop=True)
final_df.to_parquet(INTERIM_DIR / f"hmda_{year}_purchase_filtered.parquet", index=False)

print(f"Saved {len(final_df):,} rows x {final_df.shape[1]} columns")

Saved 1,727,644 rows x 15 columns


## Step 7. Summary

**Result:** 1,727,644 applications × 16 columns, saved to "data/interim/hmda_2025_purchase_filtered.parquet".

| ID | Decision |
|---|---|
| D1 | 2025 data only |
| D2 | Comparable loans only: first-lien purchases of a primary, site-built home |
| D3 | Small-lender "Exempt" removed |
| D4 | Six states, two per home-price tier |
| D5 | Download checked against the source; incomplete CA and TX files re-downloaded |

**Next:** notebook 02 cleans the data and analyses mortgage prices.